In [1]:
from google.cloud import bigquery
client = bigquery.Client()

query = '''
WITH FilteredData AS ( --filter data for specific values on stage, Primary category, subcategory and valuation as marked by FBM
  SELECT
    *,
    SAFE_CAST(Valuation_Value AS FLOAT64) AS NumericValuation
  FROM
    `proj-docai-dev.sales_rec_demo.cc_data_total`
  WHERE
    SAFE_CAST(Valuation_Value AS FLOAT64) > 1000000
    AND Stage IN ("Biddate Set", "Construction Documents", "General Contractor Award", "Low Bids Announced", "SUBBIDS: ASAP", "Construction Underway", "Post Bid")
    AND ParentCategories_PrimaryCategoryName IN (
      "Airport", "Apartments", "Auditoriums", "Bank", "College", "University", "Condominiums", "Courthouses", "Dormitories", "Elementary, Pre Schools", "Fire and Police Stations", "Food Stores", "Government - Misc. Bldgs.", "Government Offices", "High Schools", "Hospitals, Clinics", "Hotels", "Junior High Schools", "Libraries", "Medical Offices", "Military - Misc.", "Military Housing", "Military Offices", "Museums", "Nursing Homes", "Offices", "Post Offices", "Prisons", "Religious Auditoriums", "Rental Warehouses", "Restaurants", "Retail Stores", "Shopping Centers", "Special, Vocational Schools", "Sports Arenas/Convention Centers", "Warehouses", "Athletic Bldgs", "Automotive", "Cafeterias", "Clubs, Community Centers", "Entertainment", "Golf Course / Country Club", "Laboratories", "Transportation Terminals", "Water and Sewage Treatment Plants"
    )
    AND EXISTS (SELECT 1 FROM UNNEST(ParentCategories_ParentCategory) AS pc, UNNEST(pc.`ns0:SubCategories`.`ns0:SubCategory`) as sub WHERE sub IN ("Airport","Offices","Transportation Terminals","Rental Warehouses","Museums","Sports Arenas/Convention Centers","Libraries","Auditoriums","Religious Auditoriums","Elementary, Pre Schools, High Schools, Junior High Schools, Special, Vocational Schools","College, University","Clubs, Community Centers", "Athletic Bldgs","Cafeterias","Dormitories","Courthouses","Fire and Police Stations","Prisons","Government - Misc. Bldgs.","Government Offices","Post Offices","Broadcast Studios","Warehouses","Laboratories","Hospitals, Clinics","Medical Offices","Manufacturing"))
)
SELECT * EXCEPT(row_num) FROM (
  SELECT *, ROW_NUMBER() OVER (PARTITION BY ProjectID ORDER BY timeCreated DESC) as row_num
  FROM FilteredData
) WHERE row_num = 1
'''

df = client.query(query).to_dataframe()
df

,ProjectID,DataSourceID,Title,Stage,URL,UpdateDate,IsProspective,UpdateText,Valuation_Value,Valuation_Currency,...,Details_Detail_ContractConditions,Details,Details_Detail_Subbids,Details_Detail_Quantity_Unitprices,Parameters_Parameter_SingleTradeClassification,timeCreated,sourceFile,Notes_Note,Details_Detail_Status,NumericValuation
0,1007331823,US,Rocksprings ISD Additions & Renovations (CMAR),Post Bid,http://insight.cmdgroup.com/SingleSignOn/Proje...,2024-10-14,False,Updated to Post Bid stage,9500000.00,USD,...,[],<NA>,[],[],None,2024-12-02T20:24:36.284Z,1.4_Adhoc_DL_FBMSales_XML_20241125_1.xml,[],[],9500000.0
1,1006799186,US,Refurbishment of Air Handling Units,General Contractor Award,http://insight.cmdgroup.com/SingleSignOn/Proje...,2024-06-14,False,"Project reviewed, Stage confirmed as General C...",10768972.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:12:41.367Z,1.4_Adhoc_DL_FBMSales_XML_20241012_13.xml,[],[],10768972.0
2,1006842727,US,Commerce Crossroads Logistics Park,Construction Underway,http://insight.cmdgroup.com/SingleSignOn/Proje...,2023-10-09,False,Project Details or Scope was Added/Updated,100000000.00,USD,...,[],<NA>,[],[],None,2024-12-02T20:24:36.500Z,1.4_Adhoc_DL_FBMSales_XML_20241125_23.xml,[],[],100000000.0
3,1006857059,US,Granite Fuel / Oak Ridge North,Construction Documents,http://insight.cmdgroup.com/SingleSignOn/Proje...,2023-08-09,False,"Fehr, Grossman, and Cox Architects was added a...",4900000.00,USD,...,[],<NA>,[],[],None,2024-12-02T20:24:36.500Z,1.4_Adhoc_DL_FBMSales_XML_20241125_23.xml,[],[],4900000.0
4,1006862941,US,Reunion Parkway Phase 2 Project,General Contractor Award,http://insight.cmdgroup.com/SingleSignOn/Proje...,2024-03-14,False,"Project reviewed, Stage confirmed as General C...",31961800.00,USD,...,[],<NA>,[],[],None,2024-12-02T17:00:48.561Z,1.4_Adhoc_DL_FBMSales_XML_20241014_7.xml/1.4_A...,[],[],31961800.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57294,1002412012,US,Harvest Ridge Assembly of God Church Addition,Construction Underway,http://insight.cmdgroup.com/SingleSignOn/Proje...,2020-01-26,False,Updated to Construction Underway stage,3000000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:04:47.848Z,1.4_Adhoc_DL_FBMSales_XML_20241004_10.xml,[],[],3000000.0
57295,1002446271,US,Jefferson Elementary School Expansion - Norwal...,Construction Underway,http://insight.cmdgroup.com/SingleSignOn/Proje...,2022-02-02,False,Antinozzi & Associates was added as Architect,33000000.00,USD,...,[],<NA>,[],[],None,2024-12-02T20:24:37.826Z,1.4_Adhoc_DL_FBMSales_XML_20241125_70.xml,[],[],33000000.0
57296,1004469530,US,Construction of the Hereford Civic Center,General Contractor Award,http://insight.cmdgroup.com/SingleSignOn/Proje...,2022-09-20,False,City of Hereford was added as Owner,5407165.00,USD,...,[],<NA>,[],[],None,2024-12-02T20:24:37.769Z,1.4_Adhoc_DL_FBMSales_XML_20241125_69.xml,[],[],5407165.0
57297,1004867581,US,The Leonard Mixed-Use / Muskegon,Construction Underway,http://insight.cmdgroup.com/SingleSignOn/Proje...,2020-11-17,False,"Project reviewed, Stage confirmed as Construct...",9200000.00,USD,...,[],<NA>,[],[],None,2024-12-01T21:12:44.533Z,1.4_Adhoc_DL_FBMSales_XML_20241014_146.xml,[],[],9200000.0


In [23]:
# Dictionary to store the value_counts DataFrames
value_counts_dfs = {}

for col in ['Stage', 'ParentCategories_PrimaryCategoryName', 'State']:
    if col == 'State':
        # For State column, create DataFrame from address state counts
        counts = df['Addresses_Address'].apply(lambda x: x[0]['ns0:StateProvince']).value_counts(normalize=True)
        value_counts_dfs[col] = pd.DataFrame(counts).reset_index()
        value_counts_dfs[col].columns = [col, 'Proportion']
    else:
        # For other columns, create DataFrame from direct value counts
        counts = df[col].value_counts(normalize=True)
        value_counts_dfs[col] = pd.DataFrame(counts).reset_index()
        value_counts_dfs[col].columns = [col, 'Proportion']


In [24]:
import pandas as pd

# Create the sample DataFrame
df_sample = df.sample(200, random_state=42)

for col in ['Stage', 'ParentCategories_PrimaryCategoryName', 'State']:
    if col == 'State':
        # For State column, create DataFrame from address state counts
        counts = df_sample['Addresses_Address'].apply(lambda x: x[0]['ns0:StateProvince']).value_counts(normalize=True)
        value_counts_dfs[f'{col}_sample'] = pd.DataFrame(counts).reset_index()
        value_counts_dfs[f'{col}_sample'].columns = [col, 'Proportion']
    else:
        # For other columns, create DataFrame from direct value counts
        counts = df_sample[col].value_counts(normalize=True)
        value_counts_dfs[f'{col}_sample'] = pd.DataFrame(counts).reset_index()
        value_counts_dfs[f'{col}_sample'].columns = [col, 'Proportion']

In [33]:
for col in ['Stage', 'ParentCategories_PrimaryCategoryName', 'State']:
    # Merge the DataFrames for the full and
    comparison = value_counts_dfs[col].merge(value_counts_dfs[f'{col}_sample'], on=col, suffixes=('', '_sample'))
    comparison['Proportion_diff'] = comparison['Proportion'] - comparison['Proportion_sample']
    print(f'Comparison of {col} value counts:')
    print(comparison)
    print('\n')

    print(f'Summary statistics for the difference in proportions:')
    print(comparison['Proportion_diff'].apply(lambda x: abs(x)).describe())
    print('\n---------------------------------------------------\n')

Comparison of Stage value counts:
                      Stage  Proportion  Proportion_sample  Proportion_diff
0     Construction Underway    0.305782              0.320        -0.014218
1  General Contractor Award    0.278068              0.270         0.008068
2                  Post Bid    0.211906              0.205         0.006906
3    Construction Documents    0.126110              0.135        -0.008890
4        Low Bids Announced    0.072968              0.070         0.002968


Summary statistics for the difference in proportions:
count    5.000000
mean     0.008210
std      0.004055
min      0.002968
25%      0.006906
50%      0.008068
75%      0.008890
max      0.014218
Name: Proportion_diff, dtype: float64

---------------------------------------------------

Comparison of ParentCategories_PrimaryCategoryName value counts:
   ParentCategories_PrimaryCategoryName  Proportion  Proportion_sample  \
0                               Offices    0.129025              0.115   
1    